<a href="https://colab.research.google.com/github/asigatchov/vball-net-pytorch/blob/main/train_vball_net_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Загрузка датасета


In [15]:
!rm -rf /content/*

In [16]:
!ls -l; pwd

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
total 0
pwd: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


In [17]:
%cd /content
!apt install git && git clone https://github.com/asigatchov/vball-net-pytorch.git
%cd /content/vball-net-pytorch

/content
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Cloning into 'vball-net-pytorch'...
remote: Enumerating objects: 276, done.
remote: Counting objects: 100% (276/276), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 276 (delta 175), reused 198 (delta 100), pack-reused 0 (from 0)
Receiving objects: 100% (276/276), 6.68 MiB | 19.88 MiB/s, done.
Resolving deltas: 100% (175/175), done.
/content/vball-net-pytorch


Распаковываем даатсет

In [18]:
!wget https://demo.vb-ai.ru/outputs/volleyball-split.tar
!tar -xf volleyball-split.tar
!rm volleyball-split.tar

--2026-06-10 10:27:40--  https://demo.vb-ai.ru/outputs/volleyball-split.tar
Resolving demo.vb-ai.ru (demo.vb-ai.ru)... 171.22.180.112
Connecting to demo.vb-ai.ru (demo.vb-ai.ru)|171.22.180.112|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 409927680 (391M) [application/x-tar]
Saving to: ‘volleyball-split.tar’

volleyball-split.ta 100%[===================>] 390.94M  10.3MB/s    in 40s     

2026-06-10 10:28:22 (9.77 MB/s) - ‘volleyball-split.tar’ saved [409927680/409927680]



In [19]:
!pip install -U pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install opencv-python pandas scipy tqdm tensorboard matplotlib seaborn requests av fvcore lion-pytorch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://download.pytorch.org/whl/cu124
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 38.9 MB/s  0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61443 sha256=47168065d2c6cae14b1f8c9f74efdf198197858dfd48b93e6e6d4f6742b8173a
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31596 sha256=fa977d13f1d71d21c72a

In [24]:

from pathlib import Path
# Куда сохранить split train/val/test
SPLIT_DATA = "/content/vball-net-pytorch/volleyball-split"

# Куда сохранить подготовленные grid-данные
PREP_DATA = "/content/vball-net-pytorch/datasets"
OUTPUTS_DIR = "/content/vball-net-pytorch/outputs"

# Режим обучения:
# None -> обучение с нуля
# путь до best.pth -> дообучение
RESUME_CKPT = "/content/drive/MyDrive/vball_work/outputs/VballNetGridV1b_seq9_grayscale_20260510_214641/checkpoints/best.pth"
# RESUME_CKPT = None

MODEL_NAME = "VballNetGridV1b"
SEQ = 9
GRAYSCALE = True

BATCH_SIZE = 8
WORKERS = 4
LR = 0.001

In [ ]:
!python src/video_to_heatmap.py --source "$SPLIT_DATA/val"  --output "$PREP_DATA/val" \
  --mode grid \
  --force;

In [ ]:
!python src/video_to_heatmap.py --source "$SPLIT_DATA/train"  --output "$PREP_DATA/train" \
  --mode grid \
  --force;



🏸 Badminton Dataset Preprocessor
📦 OpenCV 4.13.0
📦 NumPy 2.0.2
📦 Pandas 2.2.2
📂 Source: /content/vball-net-pytorch/volleyball-split/train
📂 Output: /content/vball-net-pytorch/datasets/train
🧩 Mode: grid
⏭️  Frame step: 1
OK: Found 10 match directories, 54 videos, 54 annotation files
Completed g_beach_mix_20260507: 6 sequences, 2018 frames
Completed g_woman_pobeda_20251020_g1: 4 sequences, 1745 frames
Completed g_4m2g_transhmash_20260508: 6 sequences, 1310 frames
Completed g_4m2g_transhmash_20260424: 7 sequences, 2337 frames
Completed g_woman_transhmash_noisy_20260513: 6 sequences, 1773 frames
Completed g_man_beach_20260518: 4 sequences, 861 frames
Completed beach_bl_night_20250825: 6 sequences, 1239 frames
Processing frames:  81% 12169/15098 [06:29<01:30, 32.39frame/s]

In [ ]:
!echo python src/train_grid.py \
  --data "$PREP_DATA/train" \
  --val_data "$PREP_DATA/val" \
  --model_name "$MODEL_NAME" \
  --seq "$SEQ" \
  --grayscale \
  --no-amp \
  --epochs "$EPOCHS" \
  --batch "$BATCH_SIZE" \
  --optimizer AdamW \
  --lr "$LR" \
  --workers "$WORKERS" \
  --out "$OUTPUTS_DIR"